In [13]:
import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import log_loss, mean_absolute_error, root_mean_squared_error
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# 1. LOAD DATA
df = pd.read_excel("DataSets/Competition_Dataset_Final.xlsx")

# ==============================================================================
# SECTION 5.1: PLANT-SIDE FAILURE & COST PREDICTION (CALIBRATED BOOSTING)
# ==============================================================================
print("=== 5.1 PLANT-SIDE FAILURE & COST PREDICTION ===")

# 1. Reset cached cost column and map ground-truth warranty costs cleanly ($0 for healthy units)
# Reset cached cost column
df.drop(columns=["field_cost"], errors="ignore", inplace=True)

# 1. Target class is the exact Failure Category column (Wiring, Leak, Controls)
y_class = df["Failure Category"].fillna("No Failure").astype(str)

# 2. Ground-truth cost from your dataset's actual cost columns
# (Verify in Section 2.3 whether total field cost is Repair Cost Field Service + Warranty Cost, or just one of them)
df["field_cost"] = (
    df["Repair Cost Field Service"].fillna(0.0) +
    df["Warranty Cost"].fillna(0.0)
)
y_cost = df["field_cost"].values


# 2. Strict plant-side feature whitelist (guards against post-ship data leakage)
PLANT_FEATURES = [
    "Facility",
    "Coil Supplier",
    "Shift",
    "Test Station",
    "Product Family",
    "Controls Revision"
]
plant_cols = [c for c in PLANT_FEATURES if c in df.columns]
X = df[plant_cols].copy()

# 3. Grouping key to prevent batch/temporal leakage across folds

    # Option A: Group by manufacturing line/station + shift (creates granular operational groups)
groups = (
        df["Facility"].astype(str) + "_" +
        df["Shift"].astype(str) + "_" +
        df["Test Station"].astype(str)
)


# 4. Regularized & probability-calibrated HistGradientBoosting pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), plant_cols)
    ]
)

base_hgb = HistGradientBoostingClassifier(
    max_depth=4,
    learning_rate=0.05,
    max_iter=150,
    min_samples_leaf=20,
    l2_regularization=1.0,
    random_state=42
)

calibrated_model = Pipeline([
    ("prep", preprocessor),
    ("clf", CalibratedClassifierCV(estimator=base_hgb, method="sigmoid", cv=3))
])

# 5. Out-of-fold Grouped Cross-Validation
sgkf = StratifiedGroupKFold(n_splits=5)
classes = np.sort(y_class.unique())
oof_probs = np.zeros((len(df), len(classes)))
oof_expected_cost = np.zeros(len(df))

fold_metrics = []
for fold, (tr_idx, te_idx) in enumerate(sgkf.split(X, y_class, groups=groups)):
    X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
    y_tr, y_te = y_class.iloc[tr_idx], y_class.iloc[te_idx]
    cost_tr, cost_te = y_cost[tr_idx], y_cost[te_idx]
    
    calibrated_model.fit(X_tr, y_tr)
    probs = calibrated_model.predict_proba(X_te)
    
    # Align probability columns with sorted global classes
    aligned_probs = np.zeros((len(te_idx), len(classes)))
    for i, c in enumerate(calibrated_model.classes_):
        col_idx = np.where(classes == c)[0][0]
        aligned_probs[:, col_idx] = probs[:, i]
        
    oof_probs[te_idx] = aligned_probs
    
    # Expected Cost = Sum_k P(Class = k | X_plant) * E[Cost | Class = k, train]
    fold_class_costs = np.array([
        cost_tr[y_tr == c].mean() if (y_tr == c).any() else 0.0
        for c in classes
    ])
    pred_cost = aligned_probs @ fold_class_costs
    oof_expected_cost[te_idx] = pred_cost
    
    fold_metrics.append({
        "log_loss": log_loss(y_te, aligned_probs, labels=classes),
        "cost_mae": mean_absolute_error(cost_te, pred_cost),
        "cost_rmse": root_mean_squared_error(cost_te, pred_cost)
    })

print("Out-of-Fold Grouped CV Performance (5-Fold Mean):")
print(pd.DataFrame(fold_metrics).mean().round(4), "\n")

df["pred_expected_cost"] = oof_expected_cost
for i, c in enumerate(classes):
    df[f"prob_{c}"] = oof_probs[:, i]


# ==============================================================================
# SECTION 5.2: UNIT RISK SCORING BY CONFIGURATION
# ==============================================================================
print("=== 5.2 CONFIGURATION RISK SCORING TABLE ===")

config_cols = [c for c in ["Facility", "Coil Supplier", "Controls Revision", "Product Family"] if c in df.columns]

agg_dict = {
    "unit_volume": ("pred_expected_cost", "count"),
    "actual_avg_cost": ("field_cost", "mean"),
    "predicted_unit_risk_cost": ("pred_expected_cost", "mean"),
    "total_expected_liability": ("pred_expected_cost", "sum")
}

leak_col = next((c for c in df.columns if c.lower() == "prob_leak"), None)
if leak_col:
    agg_dict["prob_leak"] = (leak_col, "mean")

#if "prob_Leak" in df.columns:
    agg_dict["prob_leak"] = ("prob_Leak", "mean")

config_risk = (
    df.groupby(config_cols, dropna=False)
    .agg(**agg_dict)
    .reset_index()
    .sort_values("total_expected_liability", ascending=False)
)

# Dollar-based risk tiers (so $0 healthy configurations stay Low Risk)
config_risk["risk_tier"] = pd.cut(
    config_risk["predicted_unit_risk_cost"],
    bins=[-np.inf, 50.0, 250.0, np.inf],
    labels=["Low Risk", "Medium Risk", "High Risk"]
)

# 1. Assign calibrated dollar-based risk tiers
config_risk["risk_tier"] = pd.cut(
    config_risk["predicted_unit_risk_cost"],
    bins=[-np.inf, 2500.0, 5500.0, np.inf],
    labels=["Low Risk", "Medium Risk", "High Risk"]
)

# 2. Sort so High Risk appears at the top, grouped cleanly by Facility and Supplier
config_risk = config_risk.sort_values(
    by=["risk_tier", "Facility", "Coil Supplier", "total_expected_liability"],
    ascending=[False, True, True, False]
).reset_index(drop=True)


print(config_risk.to_string(index=False), "\n")

=== 5.1 PLANT-SIDE FAILURE & COST PREDICTION ===
Out-of-Fold Grouped CV Performance (5-Fold Mean):
log_loss        0.5032
cost_mae     2726.8425
cost_rmse    3859.3981
dtype: float64 

=== 5.2 CONFIGURATION RISK SCORING TABLE ===
  Facility Coil Supplier Controls Revision Product Family  unit_volume  actual_avg_cost  predicted_unit_risk_cost  total_expected_liability  prob_leak   risk_tier
Facility 1    Supplier B             Rev C        LegacyX           23     11871.347826               6999.854342             160996.649872   0.604399   High Risk
Facility 1    Supplier B             Rev C          DC300           24      6813.083333               6602.900390             158469.609361   0.565264   High Risk
Facility 1    Supplier B             Rev A          DC300           24      6298.208333               6566.859597             157604.630339   0.503475   High Risk
Facility 1    Supplier B             Rev B          DC100           21      6033.142857               6363.037942     

In [12]:
print(df["Failure Category"].value_counts(dropna=False))
print()
print(df.groupby("Failure Category")[["Repair Cost Field Service", "Warranty Cost"]].mean())
print()
# Quick check: see if test_diagnosis or Defect Category is a 1-to-1 giveaway of Failure Category
for col in ["test_diagnosis", "Defect Category", "Failure Mode Rework", "Disposition Code"]:
    if col in df.columns:
        print(f"--- {col} vs Failure Category ---")
        print(pd.crosstab(df[col], df["Failure Category"]), "\n")

Failure Category
Wiring      1929
Leak         788
Controls     283
Name: count, dtype: int64

                  Repair Cost Field Service  Warranty Cost
Failure Category                                          
Controls                        1386.883392    2774.236749
Leak                            3670.073604    7340.640863
Wiring                           280.723173     561.919647

--- Defect Category vs Failure Category ---
Failure Category  Controls  Leak  Wiring
Defect Category                         
Brazing                  0   788       0
Controls               283     0       0
Wiring                   0     0    1929 

--- Failure Mode Rework vs Failure Category ---
Failure Category     Controls  Leak  Wiring
Failure Mode Rework                        
Fan Rotation Error          0     0     181
Loose Terminal              0     0    1748
Refrigerant Leak            0   788       0
Sensor Miswire            283     0       0 

--- Disposition Code vs Failure Category ---

## Section 5: predicting future field failures and warranty costs BEFORE a unit leaves factory by looking at plant-side attributes

## 5.1
Grouping all units from facility, test station, and shift stay together in test set
- Warranty Cost is an generated as a fixed 2X multiplier of repair cost field service
- log_loss measures calibrated predicted failure probabilities across all failure categories, shows strong predictive signal for which failure mode will occur
- cost_mae (Mean Absolute Error) gap between what the model predicted and actual cost
    - come from adding average of Repair Cost Field Service and Warranty Cost
    - so average across all 3,000 units is $2,726.84

- cost_rmse (root mean squared) gives extra weight to large misses to show gap between small claim and big claim (ex wiring vs leak)

## 5.2
- Close alignment between predicted and actual group costs
Grouped by : "Facility", "Coil Supplier", "Controls Revision", "Product Family"
- NOTE: FACILITY 2 has the Highest Risk configurations and doesn't produce any ouput at a low-risk

## Average Cost by Failure Mode
- Leak is the most expensive to repair, and the most failures occur from wiring but the cheapest repair

